In [21]:
import pandas as pd
import re

In [22]:
import re
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process

engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

kodepos_df = pd.read_sql("SELECT kode, kodepos FROM wilayah_kodepos.wilayah_kodepos", engine)
kodepos_df["kodepos"] = kodepos_df["kodepos"].astype(str).str.strip()

print(wilayah.shape, kodepos_df.shape)
wilayah.head()

def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

CORE_ADMIN_WORDS = {"KABUPATEN", "KAB", "KOTA", "KECAMATAN", "KEC", "DESA", "KELURAHAN", "KEL", "ADMINISTRASI", "ADM"}

def core_name(nama):
    tokens = str(nama).replace(".", " ").split()
    return " ".join(t for t in tokens if t not in CORE_ADMIN_WORDS).strip()

wilayah["level"] = wilayah["kode"].apply(get_level)
df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

for df_ in (df_prov, df_kabkota, df_kec, df_desa):
    df_["nama_core"] = df_["nama"].apply(core_name)

print(f"provinsi={len(df_prov)}, kabkota={len(df_kabkota)}, kecamatan={len(df_kec)}, desa={len(df_desa)}")
df_prov[["nama", "nama_core"]].head()

def sequence_in_tokens(cand_tokens, text_tokens):
    n, m = len(cand_tokens), len(text_tokens)
    if n == 0 or n > m:
        return False
    for i in range(m - n + 1):
        if text_tokens[i:i+n] == cand_tokens:
            return True
    return False

(91599, 2) (83762, 2)
provinsi=38, kabkota=514, kecamatan=7285, desa=83762


In [23]:
FUZZY_THRESHOLD = 95

def fuzzy_match_one(text, candidates_df, score_cutoff=FUZZY_THRESHOLD):
    if candidates_df.empty:
        return None, None, 0
    match = process.extractOne(text, candidates_df["nama"], scorer=fuzz.token_set_ratio, score_cutoff=score_cutoff)
    if not match:
        return None, None, 0
    nama, score, idx = match
    row = candidates_df.loc[idx]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    return row["nama"], row["kode"], score

In [24]:
STRICT_THRESHOLD = 98

GENERIC_PREFIXES = {"KABUPATEN", "KAB", "KOTA", "KECAMATAN", "KEC", "DESA", "KELURAHAN", "KEL"}

PROVINCE_ALIASES = {
    "DKI JAKARTA": "DKI JAKARTA", "DKI": "DKI JAKARTA", "JKT": "DKI JAKARTA",
    "JAWA BARAT": "JAWA BARAT", "JABAR": "JAWA BARAT",
    "JAWA TENGAH": "JAWA TENGAH", "JATENG": "JAWA TENGAH",
    "JAWA TIMUR": "JAWA TIMUR", "JATIM": "JAWA TIMUR",
    "KALIMANTAN SELATAN": "KALIMANTAN SELATAN", "KALSEL": "KALIMANTAN SELATAN",
    "KALIMANTAN TIMUR": "KALIMANTAN TIMUR", "KALTIM": "KALIMANTAN TIMUR",
    "KALIMANTAN BARAT": "KALIMANTAN BARAT", "KALBAR": "KALIMANTAN BARAT",
    "KALIMANTAN TENGAH": "KALIMANTAN TENGAH", "KALTENG": "KALIMANTAN TENGAH",
    "KALIMANTAN UTARA": "KALIMANTAN UTARA", "KALTARA": "KALIMANTAN UTARA",
    "SUMATERA UTARA": "SUMATERA UTARA", "SUMUT": "SUMATERA UTARA",
    "SUMATERA BARAT": "SUMATERA BARAT", "SUMBAR": "SUMATERA BARAT",
    "SUMATERA SELATAN": "SUMATERA SELATAN", "SUMSEL": "SUMATERA SELATAN",
    "SULAWESI SELATAN": "SULAWESI SELATAN", "SULSEL": "SULAWESI SELATAN",
    "SULAWESI UTARA": "SULAWESI UTARA", "SULUT": "SULAWESI UTARA",
    "SULAWESI TENGAH": "SULAWESI TENGAH", "SULTENG": "SULAWESI TENGAH",
    "SULAWESI TENGGARA": "SULAWESI TENGGARA", "SULTRA": "SULAWESI TENGGARA",
    "SULAWESI BARAT": "SULAWESI BARAT", "SULBAR": "SULAWESI BARAT",
    "NUSA TENGGARA BARAT": "NUSA TENGGARA BARAT", "NTB": "NUSA TENGGARA BARAT",
    "NUSA TENGGARA TIMUR": "NUSA TENGGARA TIMUR", "NTT": "NUSA TENGGARA TIMUR",
    "KEPULAUAN RIAU": "KEPULAUAN RIAU", "KEPRI": "KEPULAUAN RIAU",
    "KEPULAUAN BANGKA BELITUNG": "KEPULAUAN BANGKA BELITUNG", "BABEL": "KEPULAUAN BANGKA BELITUNG",
    "DAERAH ISTIMEWA YOGYAKARTA": "DAERAH ISTIMEWA YOGYAKARTA",
    "DIY": "DAERAH ISTIMEWA YOGYAKARTA", "YOGYA": "DAERAH ISTIMEWA YOGYAKARTA", "JOGJA": "DAERAH ISTIMEWA YOGYAKARTA",
    "PAPUA BARAT": "PAPUA BARAT", "PAPBAR": "PAPUA BARAT",
}
CITY_ALIASES = {
    "JAKARTA SELATAN": "JAKARTA SELATAN", "JAKSEL": "JAKARTA SELATAN",
    "JAKARTA PUSAT": "JAKARTA PUSAT", "JAKPUS": "JAKARTA PUSAT",
    "JAKARTA UTARA": "JAKARTA UTARA", "JAKUT": "JAKARTA UTARA",
    "JAKARTA BARAT": "JAKARTA BARAT", "JAKBAR": "JAKARTA BARAT",
    "JAKARTA TIMUR": "JAKARTA TIMUR", "JAKTIM": "JAKARTA TIMUR",
    "BANDUNG": "BANDUNG", "BDG": "BANDUNG", "SURABAYA": "SURABAYA", "SBY": "SURABAYA",
    "SEMARANG": "SEMARANG", "SMG": "SEMARANG", "MEDAN": "MEDAN", "MDN": "MEDAN",
    "MAKASSAR": "MAKASSAR", "MKS": "MAKASSAR", "PALEMBANG": "PALEMBANG", "PLB": "PALEMBANG",
    "TANGERANG": "TANGERANG", "TGRG": "TANGERANG", "TNGRG": "TANGERANG",
    "BEKASI": "BEKASI", "BKS": "BEKASI", "DEPOK": "DEPOK", "DPK": "DEPOK",
    "BOGOR": "BOGOR", "BGR": "BOGOR",
}
_ALL_ALIASES = {**CITY_ALIASES, **PROVINCE_ALIASES}
_ALIAS_KEYS_SORTED = sorted(_ALL_ALIASES.keys(), key=len, reverse=True)
_ALIAS_PATTERN = re.compile(r"\b(" + "|".join(re.escape(k) for k in _ALIAS_KEYS_SORTED) + r")\b")

def expand_abbreviations(text):
    return _ALIAS_PATTERN.sub(lambda m: _ALL_ALIASES[m.group(1)], text)

# test
print(expand_abbreviations("JAKSEL DAN JATENG"))

JAKARTA SELATAN DAN JAWA TENGAH


In [25]:
def remove_matched_text(text, matched_name):
    """
    Buang kata2 dari `matched_name` yang beneran muncul di `text`, per-kata (bukan frasa utuh).
    Fuzzy match sering matched candidate name-nya "KABUPATEN NGANJUK" padahal teks aslinya
    cuma ada kata "NGANJUK" doang -> kalau strip literal frasa utuh, gak ketemu & gak kehapus
    -> kata itu bisa "ke-reuse" di level berikutnya (double match).
    """
    if not matched_name:
        return text
    words = [w for w in re.split(r'\s+', matched_name.strip()) if w and w.upper() not in GENERIC_PREFIXES]
    cleaned = text
    for w in words:
        pattern = r'\b' + re.escape(w) + r'\b'
        cleaned = re.sub(pattern, ' ', cleaned, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', cleaned).strip()

In [26]:
def common_kode_prefix(kode_list):
    """
    Cari level kode (prov/kab/kec/desa) yang SAMA di semua kandidat kode.
    len==1 -> cuma prov sama, len==2 -> prov+kab sama, len==3 -> +kec sama, dst.
    """
    splits = [k.split(".") for k in kode_list]
    max_level = min(len(s) for s in splits)
    common = []
    for i in range(max_level):
        vals = set(s[i] for s in splits)
        if len(vals) == 1:
            common.append(splits[0][i])
        else:
            break
    return common

In [27]:
def find_desa_bigram_match(tokens, desa_pool, min_len=6):
    """
    Cek apakah ada 2 token BERURUTAN di `tokens` yang kalau digabung tanpa spasi
    match EXACT ke salah satu nama desa di `desa_pool`.
    Nangkep kasus "PACE KULON" (2 kata di teks user) vs "PACEKULON" (1 kata di data),
    yang tanpa ini bakal ke-grab duluan sama level kecamatan (krn "PACE" match kecamatan).
    Return (nama_desa, kode_desa) atau (None, None).
    """
    if desa_pool.empty or len(tokens) < 2:
        return None, None
    names_map = {str(n).strip().upper(): (n, k) for n, k in zip(desa_pool["nama"], desa_pool["kode"])}
    for i in range(len(tokens) - 1):
        joined = (tokens[i] + tokens[i + 1]).upper()
        if len(joined) >= min_len and joined in names_map:
            return names_map[joined]
    return None, None

In [28]:
def tokenize(text):
    return re.findall(r"[A-Z0-9]+", text)

def expand_abbreviations(text):
    return _ALIAS_PATTERN.sub(lambda m: _ALL_ALIASES[m.group(1)], text)

def exact_match_safe(text_tokens, candidates_df, min_len=4, name_col="nama_core"):
    text_tokens_set = set(text_tokens)
    cand_sorted = candidates_df.assign(_len=candidates_df[name_col].str.len()).sort_values("_len", ascending=False)
    for _, row in cand_sorted.iterrows():
        core = row[name_col]
        if not core or len(core) < min_len:
            continue
        cand_tokens = tokenize(core)
        if not cand_tokens:
            continue
        if sequence_in_tokens(cand_tokens, text_tokens):
            return row["nama"], row["kode"]
        cand_nospace = "".join(cand_tokens)
        if len(cand_tokens) > 1 and cand_nospace in text_tokens_set and len(cand_nospace) >= min_len:
            return row["nama"], row["kode"]
    return None, None

# nama_kab, kode_kab = exact_match_safe(text_tokens, df_kabkota, min_len=4)
# print(nama_kab, kode_kab)

In [29]:
def match_wilayah_final(leftover_text, kodepos=None):
    # NORMALISASI: uppercase dari awal, krn tokenize() cuma nangkep [A-Z0-9]+.
    # Tanpa ini, teks lowercase/mixed-case bakal ke-skip total pas ditokenize.
    leftover_text = (leftover_text or "").strip().upper()

    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "match_path": None,
    }

    # ============================================
    # PATH A: ADA KODEPOS
    # ============================================
    if kodepos:
        kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
        if kandidat_kode:
            print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode wilayah")
            result["match_path"] = "with_postcode"

            pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode)]
            nama_d, kode_d = exact_match_safe(tokens, pool_desa, min_len=4)
            score_d = 100
            if not nama_d:
                nama_d, kode_d = find_desa_bigram_match(tokens, pool_desa)
                if nama_d:
                    score_d = 100
                    print(f"desa ketemu via bigram: {nama_d} (skor={score_d})")
            if not nama_d:
                nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, pool_desa, score_cutoff=STRICT_THRESHOLD)

            if nama_d:
                print(f"desa ketemu: {nama_d} (skor={score_d})")
                parts = kode_d.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), score_d
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), score_d
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], score_d
            else:
                # FALLBACK: desa gak ketemu, tapi kandidat_kode dari kodepos
                # kemungkinan besar semua satu kecamatan/kabkota/provinsi yang sama.
                print("desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos")
                common = common_kode_prefix(kandidat_kode)
                DERIVED_SCORE = 90  # penanda: derived dari postcode only, bukan exact/fuzzy match teks

                if len(common) >= 3:
                    kode_kec_common = ".".join(common[:3])
                    row_kec = df_kec[df_kec["kode"] == kode_kec_common]
                    if not row_kec.empty:
                        result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], kode_kec_common, DERIVED_SCORE
                        print(f"  -> kecamatan di-derive dari kodepos: {result['kecamatan']}")
                if len(common) >= 2:
                    kode_kab_common = ".".join(common[:2])
                    row_kab = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab.empty:
                        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kode_kab_common, DERIVED_SCORE
                        print(f"  -> kab/kota di-derive dari kodepos: {result['kabkota']}")
                if len(common) >= 1:
                    row_prov = df_prov[df_prov["kode"] == common[0]]
                    if not row_prov.empty:
                        result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], common[0], DERIVED_SCORE
                        print(f"  -> provinsi di-derive dari kodepos: {result['provinsi']}")

                if not common:
                    print("  -> kandidat kodepos gak sharing prefix apapun, beneran kosongin")

            return result
        else:
            print(f"kodepos '{kodepos}' tidak ditemukan di tabel -> lanjut sebagai tanpa kodepos")

In [39]:
def match_wilayah_final(leftover_text, kodepos=None):
    # NORMALISASI: uppercase dari awal, krn tokenize() cuma nangkep [A-Z0-9]+.
    # Tanpa ini, teks lowercase/mixed-case bakal ke-skip total pas ditokenize.
    leftover_text = (leftover_text or "").strip().upper()

    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "match_path": None,
    }

    # ============================================
    # PATH A: ADA KODEPOS
    # ============================================
    if kodepos:
        kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
        if kandidat_kode:
            print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode wilayah")
            result["match_path"] = "with_postcode"

            pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode)]
            nama_d, kode_d = exact_match_safe(tokens, pool_desa, min_len=4)
            score_d = 100
            if not nama_d:
                nama_d, kode_d = find_desa_bigram_match(tokens, pool_desa)
                if nama_d:
                    score_d = 100
                    print(f"desa ketemu via bigram: {nama_d} (skor={score_d})")
            if not nama_d:
                nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, pool_desa, score_cutoff=STRICT_THRESHOLD)

            if nama_d:
                print(f"desa ketemu: {nama_d} (skor={score_d})")
                parts = kode_d.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), score_d
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), score_d
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], score_d
            else:
                # FALLBACK: desa gak ketemu dari teks, tapi ada kandidat_kode dari kodepos.
                # Sebelumnya: langsung derive kec/kab/prov dari common prefix kandidat_kode,
                # TANPA validasi independen -> bahaya kalau kodepos_df ternyata gak lengkap/salah
                # (misal: kodepos yg sebenarnya overlap ke banyak desa, tapi di tabel cuma ke-input 1 row).
                #
                # FIX: sebelum percaya common prefix, cross-check dulu ke teks aslinya secara independen
                # (exact/fuzzy match kab/kota langsung dari leftover_text, TERLEPAS dari kandidat_kode).
                # Kalau match independen ini SETUJU (kode sama & skor tinggi) dgn hasil derive dari
                # kodepos, baru derive dipercaya turun sampe kecamatan & desa.
                # Kalau GAK ada match independen / gak setuju -> derive DITAHAN (jangan sampe desa),
                # supaya hasil yg gak ada bukti dari teks gak dipaksa jadi "seolah-olah pasti".
                print("desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos")
                common = common_kode_prefix(kandidat_kode)
                DERIVED_SCORE = 90          # derived dari postcode only, tanpa validasi silang
                VALIDATED_SCORE = 95        # derived dari postcode, DAN tervalidasi silang dari teks
                CROSSCHECK_THRESHOLD = 95   # ambang skor match independen kab/kota dari teks

                # --- cross-check independen: coba match kab/kota langsung dari teks ---
                # PENTING: pake token_set_ratio, BUKAN ratio biasa / fuzzy_match_one default.
                # Nama resmi di df_kabkota buat Jakarta itu "KOTA ADMINISTRASI JAKARTA SELATAN",
                # sedangkan teks alamat biasanya cuma nulis "JAKARTA SELATAN" (subset kata).
                #
                # MASALAH YG DITEMUKAN: token_set_ratio dihitung terhadap SELURUH leftover_text
                # (yg penuh noise: CYBER, TOWER, LANTAI, dst) DAN nama candidate masih mengandung
                # kata kualifikasi administratif ("KOTA", "ADMINISTRASI") yg jarang ditulis orang
                # di alamat. Kombinasi keduanya bikin skor jatuh (cth: 62.5, padahal "JAKARTA SELATAN"
                # ada literal di teks).
                #
                # FIX: strip kata kualifikasi administratif generik dari nama candidate DULU,
                # baru dibandingin. "KOTA ADMINISTRASI JAKARTA SELATAN" -> "JAKARTA SELATAN".
                from rapidfuzz import fuzz
                import re as _re

                ADMIN_QUALIFIERS = {
                    "KOTA", "KABUPATEN", "KAB", "ADMINISTRASI", "PROVINSI", "PROP",
                    "DAERAH", "KHUSUS", "IBUKOTA", "IBU", "KOTA."
                }

                def _strip_admin_qualifiers(nama):
                    words = _re.findall(r"[A-Z0-9]+", nama.upper())
                    core = [w for w in words if w not in ADMIN_QUALIFIERS]
                    # fallback: kalau semua kata ke-strip (jarang terjadi), pake nama asli
                    return " ".join(core) if core else nama

                kode_kab_common = ".".join(common[:2]) if len(common) >= 2 else None
                nama_k_txt, kode_k_txt, score_k_txt = None, None, 0

                if kode_kab_common is not None:
                    row_kab_candidate = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab_candidate.empty:
                        nama_kab_candidate = row_kab_candidate.iloc[0]["nama"]
                        nama_kab_core = _strip_admin_qualifiers(nama_kab_candidate)
                        score_k_txt = fuzz.token_set_ratio(leftover_text, nama_kab_core)
                        nama_k_txt, kode_k_txt = nama_kab_candidate, kode_kab_common
                        print(f"  [DEBUG-CROSSCHECK-v3] nama_asli={nama_kab_candidate!r}, nama_core={nama_kab_core!r}, token_set_ratio={score_k_txt}")

                validated = bool(
                    kode_kab_common
                    and nama_k_txt
                    and score_k_txt >= CROSSCHECK_THRESHOLD
                )

                if validated:
                    print(f"  -> tervalidasi silang dari teks: {nama_k_txt} (skor={score_k_txt}) cocok dgn derive kodepos")
                else:
                    print("  -> TIDAK ada validasi silang dari teks (atau gak cocok) -> derive ditahan di level kab/kota+prov saja, desa/kecamatan tetap null kecuali kandidat_kode memang cuma nunjuk 1 hasil yg konsisten")

                final_score = VALIDATED_SCORE if validated else DERIVED_SCORE

                if len(common) >= 2:
                    row_kab = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab.empty:
                        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kode_kab_common, final_score
                        print(f"  -> kab/kota di-derive dari kodepos: {result['kabkota']} (skor={final_score})")
                if len(common) >= 1:
                    row_prov = df_prov[df_prov["kode"] == common[0]]
                    if not row_prov.empty:
                        result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], common[0], final_score
                        print(f"  -> provinsi di-derive dari kodepos: {result['provinsi']} (skor={final_score})")

                # kecamatan & desa HANYA di-derive kalau validated=True,
                # supaya level yg lebih spesifik (rawan salah) gak diisi tanpa bukti independen
                if validated:
                    if len(common) >= 3:
                        kode_kec_common = ".".join(common[:3])
                        row_kec = df_kec[df_kec["kode"] == kode_kec_common]
                        if not row_kec.empty:
                            result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], kode_kec_common, final_score
                            print(f"  -> kecamatan di-derive dari kodepos: {result['kecamatan']} (skor={final_score})")
                    if len(common) >= 4:
                        kode_desa_common = ".".join(common[:4])
                        row_desa = df_desa[df_desa["kode"] == kode_desa_common]
                        if not row_desa.empty:
                            result["desa"], result["kode_desa"], result["score_desa"] = row_desa.iloc[0]["nama"], kode_desa_common, final_score
                            print(f"  -> desa di-derive dari kodepos: {result['desa']} (skor={final_score})")

                if not common:
                    print("  -> kandidat kodepos gak sharing prefix apapun, beneran kosongin")

            return result
        else:
            print(f"kodepos '{kodepos}' tidak ditemukan di tabel -> lanjut sebagai tanpa kodepos")

    # ============================================
    # PATH B: TANPA KODEPOS (atau kodepos gak valid)
    # ============================================
    result["match_path"] = "without_postcode"

    # LEVEL 1: PROVINSI (selalu global)
    nama_p, kode_p = exact_match_safe(tokens, df_prov, min_len=4)
    score_p = 100
    if not nama_p:
        nama_p, kode_p, score_p = fuzzy_match_one(leftover_text, df_prov, score_cutoff=STRICT_THRESHOLD)
    if nama_p:
        result["provinsi"], result["kode_prov"], result["score_prov"] = nama_p, kode_p, score_p
        print(f"provinsi ketemu: {nama_p} (skor={score_p})")
        leftover_text = remove_matched_text(leftover_text, nama_p)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL")

    # LEVEL 2: KAB/KOTA -> persempit kalau provinsi ketemu, else global
    kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")] if nama_p else df_kabkota
    nama_k, kode_k = exact_match_safe(tokens, kandidat_kab, min_len=4)
    score_k = 100
    if not nama_k:
        nama_k, kode_k, score_k = fuzzy_match_one(leftover_text, kandidat_kab, score_cutoff=STRICT_THRESHOLD)
    if nama_k:
        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = nama_k, kode_k, score_k
        print(f"kab/kota ketemu: {nama_k} (skor={score_k})")
        if not nama_p:
            kp_derived = kode_k.split(".")[0]
            row_prov = df_prov[df_prov["kode"] == kp_derived]
            if not row_prov.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_k
                print(f"  -> provinsi di-derive dari kab/kota: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_k)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL")

    # ---- CEK BIGRAM DESA DULU sebelum lock-in kecamatan ----
    # Ini nyegah token pendek kayak "PACE" ke-grab duluan jadi kecamatan,
    # padahal sebenernya bagian dari nama desa "PACEKULON" ("PACE KULON" di teks).
    desa_scope = df_desa[df_desa["kode"].str.startswith(kode_k + ".")] if nama_k else df_desa
    bigram_nama_d, bigram_kode_d = find_desa_bigram_match(tokens, desa_scope)

    if bigram_nama_d:
        print(f"desa ketemu duluan via bigram (sebelum cek kecamatan): {bigram_nama_d} (skor=100)")
        parts = bigram_kode_d.split(".")
        result["desa"], result["kode_desa"], result["score_desa"] = bigram_nama_d, bigram_kode_d, 100
        row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
        row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
        row_prov = df_prov[df_prov["kode"] == parts[0]]
        if not row_kec.empty:
            result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), 100
        if not result["kabkota"] and not row_kab.empty:
            result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), 100
        if not result["provinsi"] and not row_prov.empty:
            result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], 100
        return result  # udah lengkap sampe desa, gak perlu proses level 3 & 4 lagi

    # LEVEL 3: KECAMATAN -> persempit kalau kab/kota ketemu, else global
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_k + ".")] if nama_k else df_kec
    nama_c, kode_c = exact_match_safe(tokens, kandidat_kec, min_len=4)
    score_c = 100
    if not nama_c:
        nama_c, kode_c, score_c = fuzzy_match_one(leftover_text, kandidat_kec, score_cutoff=STRICT_THRESHOLD)
    if nama_c:
        result["kecamatan"], result["kode_kec"], result["score_kec"] = nama_c, kode_c, score_c
        print(f"kecamatan ketemu: {nama_c} (skor={score_c})")
        if not nama_k:
            kk_derived = ".".join(kode_c.split(".")[:2])
            row_kab = df_kabkota[df_kabkota["kode"] == kk_derived]
            if not row_kab.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kk_derived, score_c
                print(f"  -> kab/kota di-derive dari kecamatan: {result['kabkota']}")
            if not result["provinsi"]:
                kp_derived = kode_c.split(".")[0]
                row_prov = df_prov[df_prov["kode"] == kp_derived]
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_c
                    print(f"  -> provinsi di-derive dari kecamatan: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_c)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL")

    # LEVEL 4: DESA -> persempit kalau kecamatan ketemu, else global
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_c + ".")] if nama_c else df_desa
    nama_d, kode_d = exact_match_safe(tokens, kandidat_desa, min_len=4)
    score_d = 100
    if not nama_d:
        nama_d, kode_d = find_desa_bigram_match(tokens, kandidat_desa)
        if nama_d:
            score_d = 100
    if not nama_d:
        nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, kandidat_desa, score_cutoff=STRICT_THRESHOLD)
    if nama_d:
        result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
        print(f"desa ketemu: {nama_d} (skor={score_d})")
        parts = kode_d.split(".")
        if not result["kecamatan"]:
            row = df_kec[df_kec["kode"] == ".".join(parts[:3])]
            if not row.empty:
                result["kecamatan"], result["kode_kec"], result["score_kec"] = row.iloc[0]["nama"], ".".join(parts[:3]), score_d
        if not result["kabkota"]:
            row = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
            if not row.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row.iloc[0]["nama"], ".".join(parts[:2]), score_d
        if not result["provinsi"]:
            row = df_prov[df_prov["kode"] == parts[0]]
            if not row.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row.iloc[0]["nama"], parts[0], score_d
    else:
        print("desa: tidak ketemu -> kosongin aja")

    return result


# test
print("=== case 1: tanpa kodepos, tanpa detail desa (ada 'Jakarta Selatan' eksplisit di teks) ===")
print(match_wilayah_final("Cyber 2 Tower Lantai 26 Jl. H.R. Rasuna Said Blok X-5, No.13 Jakarta 12950 - Indonesia | Cyber 2 Tower Lantai 26, Jalan H.R. Rasuna Said Blok X-5 No. 13, Jakarta Selatan", kodepos=12950))

# print("\n=== case 2: dengan kodepos, tanpa detail desa (fallback ke common prefix) ===")
# print(match_wilayah_final("64472 NGANJUK JAWA TIMUR", kodepos=64472))

# print("\n=== case 3: tanpa kodepos, ADA nama desa 'pace kulon' (2 kata, lowercase) ===")
# print(match_wilayah_final("64472 NGANJUK JAWA TIMUR pace kulon jln. raya kediri nganjuk", kodepos=None))

=== case 1: tanpa kodepos, tanpa detail desa (ada 'Jakarta Selatan' eksplisit di teks) ===
kodepos '12950' -> 1 kandidat kode wilayah
desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos
  [DEBUG-CROSSCHECK-v3] nama_asli='KOTA ADMINISTRASI JAKARTA SELATAN', nama_core='JAKARTA SELATAN', token_set_ratio=100.0
  -> tervalidasi silang dari teks: KOTA ADMINISTRASI JAKARTA SELATAN (skor=100.0) cocok dgn derive kodepos
  -> kab/kota di-derive dari kodepos: KOTA ADMINISTRASI JAKARTA SELATAN (skor=95)
  -> provinsi di-derive dari kodepos: DAERAH KHUSUS IBUKOTA JAKARTA (skor=95)
  -> kecamatan di-derive dari kodepos: SETIABUDI (skor=95)
  -> desa di-derive dari kodepos: KUNINGAN TIMUR (skor=95)
{'provinsi': 'DAERAH KHUSUS IBUKOTA JAKARTA', 'kode_prov': '31', 'score_prov': 95, 'kabkota': 'KOTA ADMINISTRASI JAKARTA SELATAN', 'kode_kabkota': '31.74', 'score_kabkota': 95, 'kecamatan': 'SETIABUDI', 'kode_kec': '31.74.02', 'score_kec': 95, 'desa': 'KUNINGAN TIMUR', 'kode_desa': '

In [47]:
# ============================================================
# TEST HARNESS: akurasi match_wilayah_final vs ground truth
# ============================================================
# CARA PAKAI:
# 1. Pastikan function match_wilayah_final (+ semua dependency-nya:
#    kodepos_df, df_prov, df_kabkota, df_kec, df_desa, tokenize,
#    expand_abbreviations, exact_match_safe, fuzzy_match_one,
#    find_desa_bigram_match, common_kode_prefix, remove_matched_text,
#    STRICT_THRESHOLD) udah ke-load di script/notebook yg sama.
# 2. Tinggal import/paste TEST_CASES di bawah, lalu panggil run_accuracy_test().
#
# CATATAN GROUND TRUTH:
# - Ground truth dicari manual via web search (nama resmi kelurahan/
#   kecamatan/kab-kota/provinsi), BUKAN dari hasil parsing algoritma.
# - kodepos di beberapa case aku isi None krn teks aslinya emang gak nyebut
#   kodepos apapun -- kalau di data asli lo ada kolom kodepos terpisah utk
#   row2 ini, GANTI None dgn kodepos aslinya biar test lebih representatif.
# - Case yg infonya bener2 gak cukup (gak ada nama kota/kec/kodepos sama
#   sekali, jalan yg namanya umum banget kayak Yos Sudarso) -> ground truth
#   di-set None semua secara SENGAJA. Kalau algoritma ngasih null juga,
#   itu correct behavior (bukan gagal).
# ============================================================

TEST_CASES = [
    {
        "text": "JL. GURAMI NO. 173",
        "kodepos": None,
        "ground_truth": {"desa": None, "kecamatan": None, "kabkota": None, "provinsi": None},
        "note": "Nama jalan generik, gak ada kota/kodepos -> genuinely ambigu, harus null semua",
    },
    {
        "text": "MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "Menara Karya Blok X-5 Kav 1-2 = gedung nyata, tapi teks gak sebut kota apapun -> tanpa kodepos susah ketemu",
    },
    {
        "text": "GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID BLOK X-5, NO.13",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "Cyber 2 Tower, sama kompleks Rasuna Said Blok X-5",
    },
    {
        "text": "GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLOK X-5 KAV.1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "variasi singkatan GD./JL. tanpa spasi",
    },
    {
        "text": "MENARA KARYA LT.23 JL.HR.RASUNA SAID BLOK X-5 KAV.1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "tanpa spasi sama sekali antar singkatan",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SAID, BLOK X-5, KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "koma ekstra sebelum BLOK",
    },
    {
        "text": "JL. KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI LINGKUNGAN-1)",
        "kodepos": None,
        "ground_truth": {"desa": None, "kecamatan": None, "kabkota": None, "provinsi": None},
        "note": "Yos Sudarso ada di puluhan kota di Indonesia, gak ada kota/kodepos -> genuinely ambigu",
    },
    {
        "text": "RUKO GREEN GARDEN BLOK A2-10 JL. WAHIDIN SUDIROHUSODO",
        "kodepos": None,
        "ground_truth": {"desa": "KEDOYA UTARA", "kecamatan": "KEBON JERUK", "kabkota": "KOTA ADMINISTRASI JAKARTA BARAT", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "Green Garden = kompleks ruko terkenal di Kedoya Utara, Kebon Jeruk, Jakarta Barat",
    },
    {
        "text": "MENARA KARYA LT.22, JL. HR. RASUNA SAID BLK X-5, KAV.1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "singkatan 'BLK' bukan 'BLOK'",
    },
    {
        "text": "JL. KEMBALI / PDAM KELURAHAN KETAPANG SAMPIT",
        "kodepos": None,
        "ground_truth": {"desa": "KETAPANG", "kecamatan": "MENTAWA BARU KETAPANG", "kabkota": "KABUPATEN KOTAWARINGIN TIMUR", "provinsi": "KALIMANTAN TENGAH"},
        "note": "Teks eksplisit sebut 'Kelurahan Ketapang' + 'Sampit' (ibukota kec. Mentawa Baru Ketapang)",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 18 JL. HR. RASUNA SAID BLOK X-2 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "TYPO SENGAJA: 'BLOK X-2' harusnya 'BLOK X-5' (Menara Karya beneran di X-5). Ini test case buat liat apakah pipeline lo overfit ke blok number yg salah ketik, bukan buat divalidasi kebenaran blok-nya",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 27 JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "lantai beda, lokasi sama",
    },
    {
        "text": "Regional Integrated Support Area (RISA), Jalan Hauling Paringin KM.69, Desa Lasung Batu",
        "kodepos": None,
        "ground_truth": {"desa": "LASUNG BATU", "kecamatan": "PARINGIN", "kabkota": "KABUPATEN BALANGAN", "provinsi": "KALIMANTAN SELATAN"},
        "note": "Teks eksplisit sebut 'Desa Lasung Batu' + 'Paringin'",
    },
    {
        "text": "GEDUNG MENARA KARYA LT.18, JL. H.R. RASUNA SAID BLOK X-5, KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "duplikat variasi spasi",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "duplikat variasi koma",
    },
    {
        "text": "Jalan Tinggiran II Luar, RT. 017 RW. 000, Tinggiran II Luar, Tamban Barito Kuala, Kalimantan Selatan, 70566.",
        "kodepos": "70566",
        "ground_truth": {"desa": "TINGGIRAN II LUAR", "kecamatan": "TAMBAN", "kabkota": "KABUPATEN BARITO KUALA", "provinsi": "KALIMANTAN SELATAN"},
        "note": "Semua level DISEBUT EKSPLISIT di teks + ada kodepos -> harusnya paling gampang buat pipeline",
    },
    {
        "text": "GD. MENARA KARYA LT. 18 JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "duplikat variasi singkatan GD.",
    },
]


def _norm(x):
    """Normalisasi buat perbandingan: None tetep None, string di-uppercase & strip."""
    if x is None:
        return None
    return str(x).strip().upper()


def run_accuracy_test(match_fn, verbose=True):
    """
    match_fn: function dgn signature match_fn(text, kodepos=None) -> dict
              yg punya key 'desa', 'kecamatan', 'kabkota', 'provinsi'
              (ini signature match_wilayah_final lo).
    """
    fields = ["desa", "kecamatan", "kabkota", "provinsi"]
    field_correct = {f: 0 for f in fields}
    record_fully_correct = 0
    results_log = []

    for i, case in enumerate(TEST_CASES, start=1):
        pred = match_fn(case["text"], kodepos=case["kodepos"])
        gt = case["ground_truth"]

        row_result = {"idx": i, "text": case["text"][:60], "per_field": {}}
        all_correct = True

        for f in fields:
            pred_val = _norm(pred.get(f))
            gt_val = _norm(gt.get(f))
            is_correct = (pred_val == gt_val)
            row_result["per_field"][f] = {
                "predicted": pred.get(f),
                "expected": gt.get(f),
                "correct": is_correct,
            }
            if is_correct:
                field_correct[f] += 1
            else:
                all_correct = False

        if all_correct:
            record_fully_correct += 1

        results_log.append(row_result)

        if verbose:
            status = "✅ FULL MATCH" if all_correct else "❌ ADA YG BEDA"
            print(f"\n[{i}] {status}")
            print(f"    text: {case['text'][:80]}")
            for f in fields:
                r = row_result["per_field"][f]
                mark = "✓" if r["correct"] else "✗"
                print(f"    {mark} {f:10s} pred={str(r['predicted']):35s} expected={r['expected']}")

    n = len(TEST_CASES)
    print("\n" + "=" * 60)
    print("AKURASI")
    print("=" * 60)
    for f in fields:
        acc = field_correct[f] / n * 100
        print(f"  {f:10s}: {field_correct[f]}/{n}  ({acc:.1f}%)")
    overall_acc = record_fully_correct / n * 100
    print(f"  {'FULL ROW':10s}: {record_fully_correct}/{n}  ({overall_acc:.1f}%)  <- semua 4 field bener sekaligus")
    print("=" * 60)

    return results_log


# ============================================================
# CARA PAKAI (uncomment & sesuaikan setelah match_wilayah_final ke-load):
# ============================================================
run_accuracy_test(match_wilayah_final)

  [DEBUG-STOPWORDS] leftover_text setelah strip generik: 'GURAMI 173'
provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL
kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL
  -> kab/kota blm ada anchor, exact+fuzzy match kecamatan GLOBAL di-skip semua (terlalu berisiko collision)
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL
  -> kecamatan blm ada anchor, exact+fuzzy match desa GLOBAL di-skip semua (cuma bigram yg boleh, itupun gak ketemu)
desa: tidak ketemu -> kosongin aja

[1] ✅ FULL MATCH
    text: JL. GURAMI NO. 173
    ✓ desa       pred=None                                expected=None
    ✓ kecamatan  pred=None                                expected=None
    ✓ kabkota    pred=None                                expected=None
    ✓ provinsi   pred=None                                expected=None
  [DEBUG-STOPWORDS] leftover_text setelah strip generik: 'KARYA 22 H R RASUNA SAID X 5 1 2'
provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL
k

[{'idx': 1,
  'text': 'JL. GURAMI NO. 173',
  'per_field': {'desa': {'predicted': None, 'expected': None, 'correct': True},
   'kecamatan': {'predicted': None, 'expected': None, 'correct': True},
   'kabkota': {'predicted': None, 'expected': None, 'correct': True},
   'provinsi': {'predicted': None, 'expected': None, 'correct': True}}},
 {'idx': 2,
  'text': 'MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID BLOK X-5 KAV. 1',
  'per_field': {'desa': {'predicted': None,
    'expected': 'KUNINGAN TIMUR',
    'correct': False},
   'kecamatan': {'predicted': None, 'expected': 'SETIABUDI', 'correct': False},
   'kabkota': {'predicted': None,
    'expected': 'KOTA ADMINISTRASI JAKARTA SELATAN',
    'correct': False},
   'provinsi': {'predicted': None,
    'expected': 'DAERAH KHUSUS IBUKOTA JAKARTA',
    'correct': False}}},
 {'idx': 3,
  'text': 'GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID BLOK X-5, NO.',
  'per_field': {'desa': {'predicted': None,
    'expected': 'KUNINGAN TIMUR',
    'correc

In [46]:
def match_wilayah_final(leftover_text, kodepos=None):
    # NORMALISASI: uppercase dari awal, krn tokenize() cuma nangkep [A-Z0-9]+.
    # Tanpa ini, teks lowercase/mixed-case bakal ke-skip total pas ditokenize.
    leftover_text = (leftover_text or "").strip().upper()

    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "match_path": None,
    }

    # ============================================
    # PATH A: ADA KODEPOS
    # ============================================
    if kodepos:
        kandidat_kode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos), "kode"].tolist()
        if kandidat_kode:
            print(f"kodepos '{kodepos}' -> {len(kandidat_kode)} kandidat kode wilayah")
            result["match_path"] = "with_postcode"

            pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode)]
            nama_d, kode_d = exact_match_safe(tokens, pool_desa, min_len=4)
            score_d = 100
            if not nama_d:
                nama_d, kode_d = find_desa_bigram_match(tokens, pool_desa)
                if nama_d:
                    score_d = 100
                    print(f"desa ketemu via bigram: {nama_d} (skor={score_d})")
            if not nama_d:
                nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, pool_desa, score_cutoff=STRICT_THRESHOLD)

            if nama_d:
                print(f"desa ketemu: {nama_d} (skor={score_d})")
                parts = kode_d.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), score_d
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), score_d
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], score_d
            else:
                # FALLBACK: desa gak ketemu dari teks, tapi ada kandidat_kode dari kodepos.
                # Sebelumnya: langsung derive kec/kab/prov dari common prefix kandidat_kode,
                # TANPA validasi independen -> bahaya kalau kodepos_df ternyata gak lengkap/salah
                # (misal: kodepos yg sebenarnya overlap ke banyak desa, tapi di tabel cuma ke-input 1 row).
                #
                # FIX: sebelum percaya common prefix, cross-check dulu ke teks aslinya secara independen
                # (exact/fuzzy match kab/kota langsung dari leftover_text, TERLEPAS dari kandidat_kode).
                # Kalau match independen ini SETUJU (kode sama & skor tinggi) dgn hasil derive dari
                # kodepos, baru derive dipercaya turun sampe kecamatan & desa.
                # Kalau GAK ada match independen / gak setuju -> derive DITAHAN (jangan sampe desa),
                # supaya hasil yg gak ada bukti dari teks gak dipaksa jadi "seolah-olah pasti".
                print("desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos")
                common = common_kode_prefix(kandidat_kode)
                DERIVED_SCORE = 90          # derived dari postcode only, tanpa validasi silang
                VALIDATED_SCORE = 95        # derived dari postcode, DAN tervalidasi silang dari teks
                CROSSCHECK_THRESHOLD = 95   # ambang skor match independen kab/kota dari teks

                # --- cross-check independen: coba match kab/kota langsung dari teks ---
                # PENTING: pake token_set_ratio, BUKAN ratio biasa / fuzzy_match_one default.
                # Nama resmi di df_kabkota buat Jakarta itu "KOTA ADMINISTRASI JAKARTA SELATAN",
                # sedangkan teks alamat biasanya cuma nulis "JAKARTA SELATAN" (subset kata).
                #
                # MASALAH YG DITEMUKAN: token_set_ratio dihitung terhadap SELURUH leftover_text
                # (yg penuh noise: CYBER, TOWER, LANTAI, dst) DAN nama candidate masih mengandung
                # kata kualifikasi administratif ("KOTA", "ADMINISTRASI") yg jarang ditulis orang
                # di alamat. Kombinasi keduanya bikin skor jatuh (cth: 62.5, padahal "JAKARTA SELATAN"
                # ada literal di teks).
                #
                # FIX: strip kata kualifikasi administratif generik dari nama candidate DULU,
                # baru dibandingin. "KOTA ADMINISTRASI JAKARTA SELATAN" -> "JAKARTA SELATAN".
                from rapidfuzz import fuzz
                import re as _re

                ADMIN_QUALIFIERS = {
                    "KOTA", "KABUPATEN", "KAB", "ADMINISTRASI", "PROVINSI", "PROP",
                    "DAERAH", "KHUSUS", "IBUKOTA", "IBU", "KOTA."
                }

                def _strip_admin_qualifiers(nama):
                    words = _re.findall(r"[A-Z0-9]+", nama.upper())
                    core = [w for w in words if w not in ADMIN_QUALIFIERS]
                    # fallback: kalau semua kata ke-strip (jarang terjadi), pake nama asli
                    return " ".join(core) if core else nama

                kode_kab_common = ".".join(common[:2]) if len(common) >= 2 else None
                nama_k_txt, kode_k_txt, score_k_txt = None, None, 0

                if kode_kab_common is not None:
                    row_kab_candidate = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab_candidate.empty:
                        nama_kab_candidate = row_kab_candidate.iloc[0]["nama"]
                        nama_kab_core = _strip_admin_qualifiers(nama_kab_candidate)
                        score_k_txt = fuzz.token_set_ratio(leftover_text, nama_kab_core)
                        nama_k_txt, kode_k_txt = nama_kab_candidate, kode_kab_common
                        print(f"  [DEBUG-CROSSCHECK-v3] nama_asli={nama_kab_candidate!r}, nama_core={nama_kab_core!r}, token_set_ratio={score_k_txt}")

                validated = bool(
                    kode_kab_common
                    and nama_k_txt
                    and score_k_txt >= CROSSCHECK_THRESHOLD
                )

                if validated:
                    print(f"  -> tervalidasi silang dari teks: {nama_k_txt} (skor={score_k_txt}) cocok dgn derive kodepos")
                else:
                    print("  -> TIDAK ada validasi silang dari teks (atau gak cocok) -> derive ditahan di level kab/kota+prov saja, desa/kecamatan tetap null kecuali kandidat_kode memang cuma nunjuk 1 hasil yg konsisten")

                final_score = VALIDATED_SCORE if validated else DERIVED_SCORE

                if len(common) >= 2:
                    row_kab = df_kabkota[df_kabkota["kode"] == kode_kab_common]
                    if not row_kab.empty:
                        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kode_kab_common, final_score
                        print(f"  -> kab/kota di-derive dari kodepos: {result['kabkota']} (skor={final_score})")
                if len(common) >= 1:
                    row_prov = df_prov[df_prov["kode"] == common[0]]
                    if not row_prov.empty:
                        result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], common[0], final_score
                        print(f"  -> provinsi di-derive dari kodepos: {result['provinsi']} (skor={final_score})")

                # kecamatan & desa HANYA di-derive kalau validated=True,
                # supaya level yg lebih spesifik (rawan salah) gak diisi tanpa bukti independen
                if validated:
                    if len(common) >= 3:
                        kode_kec_common = ".".join(common[:3])
                        row_kec = df_kec[df_kec["kode"] == kode_kec_common]
                        if not row_kec.empty:
                            result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], kode_kec_common, final_score
                            print(f"  -> kecamatan di-derive dari kodepos: {result['kecamatan']} (skor={final_score})")
                    if len(common) >= 4:
                        kode_desa_common = ".".join(common[:4])
                        row_desa = df_desa[df_desa["kode"] == kode_desa_common]
                        if not row_desa.empty:
                            result["desa"], result["kode_desa"], result["score_desa"] = row_desa.iloc[0]["nama"], kode_desa_common, final_score
                            print(f"  -> desa di-derive dari kodepos: {result['desa']} (skor={final_score})")

                if not common:
                    print("  -> kandidat kodepos gak sharing prefix apapun, beneran kosongin")

            return result
        else:
            print(f"kodepos '{kodepos}' tidak ditemukan di tabel -> lanjut sebagai tanpa kodepos")

    # ============================================
    # PATH B: TANPA KODEPOS (atau kodepos gak valid)
    # ============================================
    result["match_path"] = "without_postcode"

    # ============================================================
    # STOPWORDS: kata generik struktur bangunan/alamat yg SERING kebetulan
    # jadi nama desa/kabkota asli di tempat lain (mis. "GEDUNG" -> ada desa
    # "Gedung" di Lampung, "RUKO"/"DALAM" -> ada desa dgn nama itu persis,
    # "MENARA" -> nyambung bigram ke "KOTA MENARA" di Sulut, dst).
    #
    # Tanpa di-strip, GLOBAL search (Level 1-4 saat kodepos gak ada & belum
    # ada level manapun yg ke-lock) bisa nyangkut ke daerah yg SALAH TOTAL
    # cuma krn kebetulan nama generik struktur bangunan matching nama desa asli.
    #
    # Di-strip SEKALI di awal Path B, sebelum tokens dipakai buat cari
    # provinsi/kab/kota/kecamatan/desa manapun.
    # ============================================================
    GENERIC_ADDRESS_STOPWORDS = {
        "GEDUNG", "GD", "MENARA", "TOWER", "WISMA", "GRAHA", "PLAZA",
        "LANTAI", "LT", "BLOK", "BLK", "KAV", "KAVLING", "KAVELING",
        "RUKO", "RUKAN", "RUMAH", "KOMPLEK", "KOMP", "CLUSTER",
        "JL", "JALAN", "NO", "NOMOR", "RT", "RW", "GANG", "GG",
        "DALAM", "LUAR", "PRIBADI", "LINGKUNGAN", "PERUM", "PERUMAHAN",
    }
    tokens = [t for t in tokens if t not in GENERIC_ADDRESS_STOPWORDS]
    leftover_text = " ".join(tokens)
    print(f"  [DEBUG-STOPWORDS] leftover_text setelah strip generik: {leftover_text!r}")

    # LEVEL 1: PROVINSI (selalu global)
    nama_p, kode_p = exact_match_safe(tokens, df_prov, min_len=4)
    score_p = 100
    if not nama_p:
        nama_p, kode_p, score_p = fuzzy_match_one(leftover_text, df_prov, score_cutoff=STRICT_THRESHOLD)
    if nama_p:
        result["provinsi"], result["kode_prov"], result["score_prov"] = nama_p, kode_p, score_p
        print(f"provinsi ketemu: {nama_p} (skor={score_p})")
        leftover_text = remove_matched_text(leftover_text, nama_p)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL")

    # LEVEL 2: KAB/KOTA -> persempit kalau provinsi ketemu, else global
    kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")] if nama_p else df_kabkota
    nama_k, kode_k = exact_match_safe(tokens, kandidat_kab, min_len=4)
    score_k = 100
    if not nama_k:
        nama_k, kode_k, score_k = fuzzy_match_one(leftover_text, kandidat_kab, score_cutoff=STRICT_THRESHOLD)
    if nama_k:
        result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = nama_k, kode_k, score_k
        print(f"kab/kota ketemu: {nama_k} (skor={score_k})")
        if not nama_p:
            kp_derived = kode_k.split(".")[0]
            row_prov = df_prov[df_prov["kode"] == kp_derived]
            if not row_prov.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_k
                print(f"  -> provinsi di-derive dari kab/kota: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_k)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kab/kota: tidak ketemu -> lanjut cari kecamatan secara GLOBAL")

    # ---- CEK BIGRAM DESA DULU sebelum lock-in kecamatan ----
    # Ini nyegah token pendek kayak "PACE" ke-grab duluan jadi kecamatan,
    # padahal sebenernya bagian dari nama desa "PACEKULON" ("PACE KULON" di teks).
    desa_scope = df_desa[df_desa["kode"].str.startswith(kode_k + ".")] if nama_k else df_desa
    bigram_nama_d, bigram_kode_d = find_desa_bigram_match(tokens, desa_scope)

    if bigram_nama_d:
        print(f"desa ketemu duluan via bigram (sebelum cek kecamatan): {bigram_nama_d} (skor=100)")
        parts = bigram_kode_d.split(".")
        result["desa"], result["kode_desa"], result["score_desa"] = bigram_nama_d, bigram_kode_d, 100
        row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
        row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
        row_prov = df_prov[df_prov["kode"] == parts[0]]
        if not row_kec.empty:
            result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), 100
        if not result["kabkota"] and not row_kab.empty:
            result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), 100
        if not result["provinsi"] and not row_prov.empty:
            result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], 100
        return result  # udah lengkap sampe desa, gak perlu proses level 3 & 4 lagi

    # LEVEL 3: KECAMATAN -> persempit kalau kab/kota ketemu, else global
    # PENTING: exact_match_safe SEBELUMnya jalan GLOBAL tanpa syarat -- ternyata
    # sama bahayanya kayak fuzzy, krn banyak kecamatan/desa yg namanya cuma
    # 1 kata pendek yg kebetulan sama kata umum. Sekarang exact match GLOBAL
    # (tanpa anchor kab/kota) JUGA di-skip, bukan cuma fuzzy-nya doang.
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_k + ".")] if nama_k else df_kec
    if nama_k:
        nama_c, kode_c = exact_match_safe(tokens, kandidat_kec, min_len=4)
        score_c = 100
        if not nama_c:
            nama_c, kode_c, score_c = fuzzy_match_one(leftover_text, kandidat_kec, score_cutoff=STRICT_THRESHOLD)
    else:
        nama_c, kode_c, score_c = None, None, 0
        print("  -> kab/kota blm ada anchor, exact+fuzzy match kecamatan GLOBAL di-skip semua (terlalu berisiko collision)")
    if nama_c:
        result["kecamatan"], result["kode_kec"], result["score_kec"] = nama_c, kode_c, score_c
        print(f"kecamatan ketemu: {nama_c} (skor={score_c})")
        if not nama_k:
            kk_derived = ".".join(kode_c.split(".")[:2])
            row_kab = df_kabkota[df_kabkota["kode"] == kk_derived]
            if not row_kab.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], kk_derived, score_c
                print(f"  -> kab/kota di-derive dari kecamatan: {result['kabkota']}")
            if not result["provinsi"]:
                kp_derived = kode_c.split(".")[0]
                row_prov = df_prov[df_prov["kode"] == kp_derived]
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], kp_derived, score_c
                    print(f"  -> provinsi di-derive dari kecamatan: {result['provinsi']}")
        leftover_text = remove_matched_text(leftover_text, nama_c)
        tokens = tokenize(expand_abbreviations(leftover_text))
    else:
        print("kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL")

    # LEVEL 4: DESA -> persempit kalau kecamatan ketemu, else global
    # PENTING: sama kayak Level 3, exact_match_safe SEBELUMnya jalan GLOBAL
    # tanpa syarat & itu yg bikin "KARYA" (nama gedung) ke-exact-match ke desa
    # yg namanya kebetulan persis "Karya". Sekarang: kalau kecamatan (nama_c)
    # BELUM ada sbg anchor, cuma bigram match (2+ kata spesifik) yg boleh jalan;
    # single-token exact match & fuzzy GLOBAL di-skip krn resiko collision-nya
    # paling tinggi di level ini (desa = ~80rb+ kandidat).
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_c + ".")] if nama_c else df_desa

    if nama_c:
        nama_d, kode_d = exact_match_safe(tokens, kandidat_desa, min_len=4)
        score_d = 100
        if not nama_d:
            nama_d, kode_d = find_desa_bigram_match(tokens, kandidat_desa)
            if nama_d:
                score_d = 100
        if not nama_d:
            nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, kandidat_desa, score_cutoff=STRICT_THRESHOLD)
    else:
        # tanpa anchor kecamatan: HANYA bigram yg boleh (butuh 2 kata match
        # sekaligus -> jauh lebih presisi drpd exact/fuzzy 1 kata generik)
        nama_d, kode_d = find_desa_bigram_match(tokens, kandidat_desa)
        score_d = 100 if nama_d else 0
        if not nama_d:
            print("  -> kecamatan blm ada anchor, exact+fuzzy match desa GLOBAL di-skip semua (cuma bigram yg boleh, itupun gak ketemu)")
    if nama_d:
        result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d
        print(f"desa ketemu: {nama_d} (skor={score_d})")
        parts = kode_d.split(".")
        if not result["kecamatan"]:
            row = df_kec[df_kec["kode"] == ".".join(parts[:3])]
            if not row.empty:
                result["kecamatan"], result["kode_kec"], result["score_kec"] = row.iloc[0]["nama"], ".".join(parts[:3]), score_d
        if not result["kabkota"]:
            row = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
            if not row.empty:
                result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row.iloc[0]["nama"], ".".join(parts[:2]), score_d
        if not result["provinsi"]:
            row = df_prov[df_prov["kode"] == parts[0]]
            if not row.empty:
                result["provinsi"], result["kode_prov"], result["score_prov"] = row.iloc[0]["nama"], parts[0], score_d
    else:
        print("desa: tidak ketemu -> kosongin aja")

    return result


# test
print("=== case 1: tanpa kodepos, tanpa detail desa (ada 'Jakarta Selatan' eksplisit di teks) ===")
print(match_wilayah_final("Cyber 2 Tower Lantai 26 Jl. H.R. Rasuna Said Blok X-5, No.13 Jakarta 12950 - Indonesia | Cyber 2 Tower Lantai 26, Jalan H.R. Rasuna Said Blok X-5 No. 13, Jakarta Selatan", kodepos=12950))

# print("\n=== case 2: dengan kodepos, tanpa detail desa (fallback ke common prefix) ===")
# print(match_wilayah_final("64472 NGANJUK JAWA TIMUR", kodepos=64472))

# print("\n=== case 3: tanpa kodepos, ADA nama desa 'pace kulon' (2 kata, lowercase) ===")
# print(match_wilayah_final("64472 NGANJUK JAWA TIMUR pace kulon jln. raya kediri nganjuk", kodepos=None))

=== case 1: tanpa kodepos, tanpa detail desa (ada 'Jakarta Selatan' eksplisit di teks) ===
kodepos '12950' -> 1 kandidat kode wilayah
desa gak ketemu -> coba derive kec/kab/prov dari kesamaan kandidat kodepos
  [DEBUG-CROSSCHECK-v3] nama_asli='KOTA ADMINISTRASI JAKARTA SELATAN', nama_core='JAKARTA SELATAN', token_set_ratio=100.0
  -> tervalidasi silang dari teks: KOTA ADMINISTRASI JAKARTA SELATAN (skor=100.0) cocok dgn derive kodepos
  -> kab/kota di-derive dari kodepos: KOTA ADMINISTRASI JAKARTA SELATAN (skor=95)
  -> provinsi di-derive dari kodepos: DAERAH KHUSUS IBUKOTA JAKARTA (skor=95)
  -> kecamatan di-derive dari kodepos: SETIABUDI (skor=95)
  -> desa di-derive dari kodepos: KUNINGAN TIMUR (skor=95)
{'provinsi': 'DAERAH KHUSUS IBUKOTA JAKARTA', 'kode_prov': '31', 'score_prov': 95, 'kabkota': 'KOTA ADMINISTRASI JAKARTA SELATAN', 'kode_kabkota': '31.74', 'score_kabkota': 95, 'kecamatan': 'SETIABUDI', 'kode_kec': '31.74.02', 'score_kec': 95, 'desa': 'KUNINGAN TIMUR', 'kode_desa': '

In [49]:
# ============================================================
# TEST HARNESS: akurasi match_wilayah_final vs ground truth
# ============================================================
# CARA PAKAI:
# 1. Pastikan function match_wilayah_final (+ semua dependency-nya:
#    kodepos_df, df_prov, df_kabkota, df_kec, df_desa, tokenize,
#    expand_abbreviations, exact_match_safe, fuzzy_match_one,
#    find_desa_bigram_match, common_kode_prefix, remove_matched_text,
#    STRICT_THRESHOLD) udah ke-load di script/notebook yg sama.
# 2. Tinggal import/paste TEST_CASES di bawah, lalu panggil run_accuracy_test().
#
# CATATAN GROUND TRUTH:
# - Ground truth dicari manual via web search (nama resmi kelurahan/
#   kecamatan/kab-kota/provinsi), BUKAN dari hasil parsing algoritma.
# - kodepos di beberapa case aku isi None krn teks aslinya emang gak nyebut
#   kodepos apapun -- kalau di data asli lo ada kolom kodepos terpisah utk
#   row2 ini, GANTI None dgn kodepos aslinya biar test lebih representatif.
# - Case yg infonya bener2 gak cukup (gak ada nama kota/kec/kodepos sama
#   sekali, jalan yg namanya umum banget kayak Yos Sudarso) -> ground truth
#   di-set None semua secara SENGAJA. Kalau algoritma ngasih null juga,
#   itu correct behavior (bukan gagal).
# ============================================================

TEST_CASES = [
    {
        "text": "JL. PULOAYANG RAYA BLOK OR-1 KIP, JAKARTA TIMUR",
        "kodepos": None,
        "ground_truth": {"desa": None, "kecamatan": None, "kabkota": None, "provinsi": None},
        "note": "Nama jalan generik, gak ada kota/kodepos -> genuinely ambigu, harus null semua",
    },
    {
        "text": "MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "Menara Karya Blok X-5 Kav 1-2 = gedung nyata, tapi teks gak sebut kota apapun -> tanpa kodepos susah ketemu",
    },
    {
        "text": "GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID BLOK X-5, NO.13",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "Cyber 2 Tower, sama kompleks Rasuna Said Blok X-5",
    },
    {
        "text": "GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLOK X-5 KAV.1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "variasi singkatan GD./JL. tanpa spasi",
    },
    {
        "text": "MENARA KARYA LT.23 JL.HR.RASUNA SAID BLOK X-5 KAV.1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "tanpa spasi sama sekali antar singkatan",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SAID, BLOK X-5, KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "koma ekstra sebelum BLOK",
    },
    {
        "text": "JL. KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI LINGKUNGAN-1)",
        "kodepos": None,
        "ground_truth": {"desa": None, "kecamatan": None, "kabkota": None, "provinsi": None},
        "note": "Yos Sudarso ada di puluhan kota di Indonesia, gak ada kota/kodepos -> genuinely ambigu",
    },
    {
        "text": "RUKO GREEN GARDEN BLOK A2-10 JL. WAHIDIN SUDIROHUSODO",
        "kodepos": None,
        "ground_truth": {"desa": "KEDOYA UTARA", "kecamatan": "KEBON JERUK", "kabkota": "KOTA ADMINISTRASI JAKARTA BARAT", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "Green Garden = kompleks ruko terkenal di Kedoya Utara, Kebon Jeruk, Jakarta Barat",
    },
    {
        "text": "MENARA KARYA LT.22, JL. HR. RASUNA SAID BLK X-5, KAV.1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "singkatan 'BLK' bukan 'BLOK'",
    },
    {
        "text": "JL. KEMBALI / PDAM KELURAHAN KETAPANG SAMPIT",
        "kodepos": None,
        "ground_truth": {"desa": "KETAPANG", "kecamatan": "MENTAWA BARU KETAPANG", "kabkota": "KABUPATEN KOTAWARINGIN TIMUR", "provinsi": "KALIMANTAN TENGAH"},
        "note": "Teks eksplisit sebut 'Kelurahan Ketapang' + 'Sampit' (ibukota kec. Mentawa Baru Ketapang)",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 18 JL. HR. RASUNA SAID BLOK X-2 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "TYPO SENGAJA: 'BLOK X-2' harusnya 'BLOK X-5' (Menara Karya beneran di X-5). Ini test case buat liat apakah pipeline lo overfit ke blok number yg salah ketik, bukan buat divalidasi kebenaran blok-nya",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 27 JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "lantai beda, lokasi sama",
    },
    {
        "text": "Regional Integrated Support Area (RISA), Jalan Hauling Paringin KM.69, Desa Lasung Batu",
        "kodepos": None,
        "ground_truth": {"desa": "LASUNG BATU", "kecamatan": "PARINGIN", "kabkota": "KABUPATEN BALANGAN", "provinsi": "KALIMANTAN SELATAN"},
        "note": "Teks eksplisit sebut 'Desa Lasung Batu' + 'Paringin'",
    },
    {
        "text": "GEDUNG MENARA KARYA LT.18, JL. H.R. RASUNA SAID BLOK X-5, KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "duplikat variasi spasi",
    },
    {
        "text": "GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "duplikat variasi koma",
    },
    {
        "text": "Jalan Tinggiran II Luar, RT. 017 RW. 000, Tinggiran II Luar, Tamban Barito Kuala, Kalimantan Selatan, 70566.",
        "kodepos": "70566",
        "ground_truth": {"desa": "TINGGIRAN II LUAR", "kecamatan": "TAMBAN", "kabkota": "KABUPATEN BARITO KUALA", "provinsi": "KALIMANTAN SELATAN"},
        "note": "Semua level DISEBUT EKSPLISIT di teks + ada kodepos -> harusnya paling gampang buat pipeline",
    },
    {
        "text": "GD. MENARA KARYA LT. 18 JL. H.R. RASUNA SAID BLOK X-5 KAV. 1-2",
        "kodepos": None,
        "ground_truth": {"desa": "KUNINGAN TIMUR", "kecamatan": "SETIABUDI", "kabkota": "KOTA ADMINISTRASI JAKARTA SELATAN", "provinsi": "DAERAH KHUSUS IBUKOTA JAKARTA"},
        "note": "duplikat variasi singkatan GD.",
    },
]


def _norm(x):
    """Normalisasi buat perbandingan: None tetep None, string di-uppercase & strip."""
    if x is None:
        return None
    return str(x).strip().upper()


def run_accuracy_test(match_fn, verbose=True):
    """
    match_fn: function dgn signature match_fn(text, kodepos=None) -> dict
              yg punya key 'desa', 'kecamatan', 'kabkota', 'provinsi'
              (ini signature match_wilayah_final lo).
    """
    fields = ["desa", "kecamatan", "kabkota", "provinsi"]
    field_correct = {f: 0 for f in fields}
    record_fully_correct = 0
    results_log = []

    for i, case in enumerate(TEST_CASES, start=1):
        pred = match_fn(case["text"], kodepos=case["kodepos"])
        gt = case["ground_truth"]

        row_result = {"idx": i, "text": case["text"][:60], "per_field": {}}
        all_correct = True

        for f in fields:
            pred_val = _norm(pred.get(f))
            gt_val = _norm(gt.get(f))
            is_correct = (pred_val == gt_val)
            row_result["per_field"][f] = {
                "predicted": pred.get(f),
                "expected": gt.get(f),
                "correct": is_correct,
            }
            if is_correct:
                field_correct[f] += 1
            else:
                all_correct = False

        if all_correct:
            record_fully_correct += 1

        results_log.append(row_result)

        if verbose:
            status = "✅ FULL MATCH" if all_correct else "❌ ADA YG BEDA"
            print(f"\n[{i}] {status}")
            print(f"    text: {case['text'][:80]}")
            for f in fields:
                r = row_result["per_field"][f]
                mark = "✓" if r["correct"] else "✗"
                print(f"    {mark} {f:10s} pred={str(r['predicted']):35s} expected={r['expected']}")

    n = len(TEST_CASES)
    print("\n" + "=" * 60)
    print("AKURASI")
    print("=" * 60)
    for f in fields:
        acc = field_correct[f] / n * 100
        print(f"  {f:10s}: {field_correct[f]}/{n}  ({acc:.1f}%)")
    overall_acc = record_fully_correct / n * 100
    print(f"  {'FULL ROW':10s}: {record_fully_correct}/{n}  ({overall_acc:.1f}%)  <- semua 4 field bener sekaligus")
    print("=" * 60)

    return results_log


# ============================================================
# CARA PAKAI (uncomment & sesuaikan setelah match_wilayah_final ke-load):
# ============================================================
run_accuracy_test(match_wilayah_final)

  [DEBUG-STOPWORDS] leftover_text setelah strip generik: 'PULOAYANG RAYA OR 1 KIP JAKARTA TIMUR'
provinsi: tidak ketemu -> lanjut cari kab/kota secara GLOBAL
kab/kota ketemu: KOTA ADMINISTRASI JAKARTA TIMUR (skor=100)
  -> provinsi di-derive dari kab/kota: DAERAH KHUSUS IBUKOTA JAKARTA
kecamatan: tidak ketemu -> lanjut cari desa secara GLOBAL
  -> kecamatan blm ada anchor, exact+fuzzy match desa GLOBAL di-skip semua (cuma bigram yg boleh, itupun gak ketemu)
desa: tidak ketemu -> kosongin aja

[1] ❌ ADA YG BEDA
    text: JL. PULOAYANG RAYA BLOK OR-1 KIP, JAKARTA TIMUR
    ✓ desa       pred=None                                expected=None
    ✓ kecamatan  pred=None                                expected=None
    ✗ kabkota    pred=KOTA ADMINISTRASI JAKARTA TIMUR     expected=None
    ✗ provinsi   pred=DAERAH KHUSUS IBUKOTA JAKARTA       expected=None
  [DEBUG-STOPWORDS] leftover_text setelah strip generik: 'KARYA 22 H R RASUNA SAID X 5 1 2'
provinsi: tidak ketemu -> lanjut cari kab/kota

[{'idx': 1,
  'text': 'JL. PULOAYANG RAYA BLOK OR-1 KIP, JAKARTA TIMUR',
  'per_field': {'desa': {'predicted': None, 'expected': None, 'correct': True},
   'kecamatan': {'predicted': None, 'expected': None, 'correct': True},
   'kabkota': {'predicted': 'KOTA ADMINISTRASI JAKARTA TIMUR',
    'expected': None,
    'correct': False},
   'provinsi': {'predicted': 'DAERAH KHUSUS IBUKOTA JAKARTA',
    'expected': None,
    'correct': False}}},
 {'idx': 2,
  'text': 'MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID BLOK X-5 KAV. 1',
  'per_field': {'desa': {'predicted': None,
    'expected': 'KUNINGAN TIMUR',
    'correct': False},
   'kecamatan': {'predicted': None, 'expected': 'SETIABUDI', 'correct': False},
   'kabkota': {'predicted': None,
    'expected': 'KOTA ADMINISTRASI JAKARTA SELATAN',
    'correct': False},
   'provinsi': {'predicted': None,
    'expected': 'DAERAH KHUSUS IBUKOTA JAKARTA',
    'correct': False}}},
 {'idx': 3,
  'text': 'GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID BL